# When PINNs fail: the SIR epidemic that never ignites (a causality trap)

The **SIR** model of mathematical epidemiology splits a population into **S**usceptible, **I**nfected and **R**ecovered fractions and tracks how a disease spreads — susceptibles are infected on contact at rate $\beta$, infectives recover at rate $\gamma$:
$$\dot S=-\beta S I,\qquad \dot I=\beta S I-\gamma I,\qquad \dot R=\gamma I,\qquad S+I+R=1.$$
With $\beta=3,\ \gamma=1$ (basic reproduction number $R_0=3$) and a small seed $I(0)=0.01$, the outbreak grows, peaks near $t\approx2.7$, and subsides.

**The trap.** The *disease-free state* $I\equiv0$ (with $S$ constant) satisfies the equations *exactly* — only the initial seed distinguishes it from the real epidemic. A vanilla PINN, minimising the residual over the whole interval at once, collapses onto this trivial state: it predicts no outbreak, with a perfectly falling loss. This is the **causality violation** of §5.3.3. The cure is **causal weighting** (Wang et al.): weight each time's residual by the accumulated residual of all earlier times, so the network must resolve the ignition first.

In [ ]:
import time
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

BETA, GAMMA, T = 3.0, 1.0, 8.0            # R0 = beta/gamma = 3
S0, I0, R0 = 0.99, 0.01, 0.0
y0 = torch.tensor([S0, I0, R0], dtype=torch.float32, device=device)

# high-accuracy Runge-Kutta reference
def rhs(t, y):
    S, I, R = y
    return [-BETA*S*I, BETA*S*I - GAMMA*I, GAMMA*I]
sol = solve_ivp(rhs, [0, T], [S0, I0, R0], dense_output=True, rtol=1e-10, atol=1e-12)
tg  = np.linspace(0, T, 400); ref = sol.sol(tg)          # (3, 400): S, I, R

def dt(f, x): return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]

In [ ]:
def train(eps, epochs=20000):
    """eps=0 -> vanilla PINN; eps>0 -> causal weighting w_i = exp(-eps * sum_{j<i} L_j)."""
    torch.manual_seed(0); np.random.seed(0)
    net = nn.Sequential(nn.Linear(1,64), nn.Tanh(), nn.Linear(64,64), nn.Tanh(),
                        nn.Linear(64,64), nn.Tanh(), nn.Linear(64,3)).to(device)
    opt = torch.optim.Adam(net.parameters(), 3e-3)
    def trial(t): return y0 + t*net(t/T)                 # hard-enforces S,I,R at t=0
    t0 = time.perf_counter()
    for e in range(epochs):
        if e == int(0.5*epochs):
            for g in opt.param_groups: g['lr'] = 6e-4
        if e == int(0.8*epochs):
            for g in opt.param_groups: g['lr'] = 1e-4
        opt.zero_grad()
        t = torch.rand(1024,1, device=device)*T
        t, _ = torch.sort(t, 0)                           # ascending in time (for causal prefix sum)
        t = t.requires_grad_(True)
        y = trial(t); S, I, R = y[:,0:1], y[:,1:2], y[:,2:3]
        L = (dt(S,t) + BETA*S*I)**2 + (dt(I,t) - BETA*S*I + GAMMA*I)**2 + (dt(R,t) - GAMMA*I)**2
        if eps > 0:
            with torch.no_grad():
                w = torch.exp(-eps*(torch.cumsum(L,0) - L))   # weight front sweeps forward in time
            loss = (w*L).mean()
        else:
            loss = L.mean()
        loss.backward(); opt.step()
    if device.type == 'cuda': torch.cuda.synchronize()
    with torch.no_grad():
        tt = torch.tensor(tg, dtype=torch.float32, device=device).reshape(-1,1)
        yp = (y0 + tt*net(tt/T)).cpu().numpy().T
    rel = np.sqrt(np.mean((yp-ref)**2)/np.mean(ref**2))
    return yp, rel, time.perf_counter()-t0

yp_van, rel_van, t_van = train(0.0)      # vanilla: collapses
yp_c,   rel_c,   t_c   = train(100.0)    # causal weighting: works
print(f'vanilla PINN : rel L2 = {rel_van:.2f}   ({t_van:.0f} s)   <- collapses to disease-free I=0')
print(f'causal weight: rel L2 = {rel_c:.1e}   ({t_c:.0f} s)   <- outbreak captured')

In [ ]:
lab = ['S', 'I', 'R']; col = ['tab:blue', 'tab:red', 'tab:green']
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3), sharey=True)
for a, (yp, ttl) in zip(ax, [(yp_van, f'Vanilla PINN: collapses to disease-free state (rel L2 = {rel_van:.1f})'),
                             (yp_c,   f'Causal weighting: outbreak captured (rel L2 = {rel_c:.0e})')]):
    for k in range(3):
        a.plot(tg, ref[k], color=col[k], lw=2.6, alpha=.5, label=f'{lab[k]} (RK)')
        a.plot(tg, yp[k], '--', color=col[k], lw=1.6)
    a.set_xlabel('time'); a.set_title(ttl, fontsize=10); a.grid(alpha=.3)
ax[0].set_ylabel('population fraction'); ax[0].plot([], [], 'k--', label='PINN')
ax[0].legend(fontsize=8, ncol=2, loc='center right')
plt.tight_layout(); plt.show()

The vanilla PINN reports a perfectly falling loss while predicting that no epidemic ever happens — the classic causality collapse onto a trivial steady state. Causal weighting makes the residual at each instant count only once the earlier instants are resolved, so the network is forced to ignite the outbreak before it can settle into the tail, and recovers the SIR curves to $\sim10^{-3}$. The moral: any initial-value problem with a stable state the network can hide in is vulnerable, and the arrow of time must be put back into the loss.